In [214]:
# SRC: https://docs.langchain.com/oss/python/langchain/multi-agent/subagents-personal-assistant

In [215]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
from langgraph.graph import StateGraph, END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing_extensions import TypedDict, Literal, Annotated
from langchain.messages import HumanMessage, SystemMessage, AnyMessage
from langgraph.graph.message import add_messages
from langchain.tools import tool
from tavily import TavilyClient
from pydantic import BaseModel, Field
from pprint import pprint
from IPython.display import Markdown, display, Image
from langgraph.prebuilt import ToolNode

In [216]:
load_dotenv(".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [217]:
basic_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)
advanced_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)

In [218]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [219]:
# Prompts - Read Content from Files
f = open("./prompts/fetching.md")
FETCHING_PROMPT = f.read()
f = open("./prompts/generations.md")
GENERATION_PROMPT = f.read()
f = open("./prompts/optimisation.md")
OPTIMISATION_PROMPT = f.read()
f = open("./prompts/scoring.md")
SCORING_PROMPT = f.read()
f = open("./prompts/validation.md")
VALIDATION_PROMPT = f.read()

In [236]:
@tool
def getDataFromExcelFile(customerName: str):
    """
        Retrieves customer records from the Excel file.
        Args:
            customerName: The name of the customer to look up in execel file.
        Return:
            return the customer data or no Data available here.
    """
    # Logic When data is Ready (pandas)
    return "No Data Available Here."

fetcher_agent = create_agent(
    model=basic_llm,
    tools=[getDataFromExcelFile],
    system_prompt=FETCHING_PROMPT
)

client_data = """
    generate marketing offre. for my client ahmed.
"""

class CustomerData(TypedDict):
    customer_data: str

query = f"USER_PROMPT = {client_data}"
clean_customer_data = ""
try:
    customer_data = fetcher_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
    clean_customer_data = customer_data.get("messages")[-1].content
    print(clean_customer_data)
except:
    clean_customer_data = "no data available"
    print("ERROR: ", clean_customer_data)

No Data Available


In [237]:
# Generation Agent
@tool()
def check_policy_rules():
    """
    Validates offer compliance against company promotional guidelines.
    """
    return "no policy now!."

offer_agent = create_agent(
    model=basic_llm,
    tools=[check_policy_rules],
    system_prompt=GENERATION_PROMPT
)

query = f"""
    USER_PROMPT = {client_data} \n\n
    Customer Data = {clean_customer_data}

"""
offre = offer_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
offre = Markdown(offre.get("messages")[-1].content)
offre

Dear Ahmed,

Welcome to our community! We're excited to have you on board. As a valued member, we'd like to offer you a special welcome promotion.

**Get 10% off your first purchase**

Use code WELCOME10 at checkout to redeem your discount. This offer is exclusive to new customers and can be used on any product in our store.

**Free Shipping on Orders Over $50**

We're committed to making your shopping experience as smooth as possible. That's why we're offering free shipping on all orders over $50.

**Stay Up-to-Date with the Latest News and Offers**

Follow us on social media to stay informed about new products, promotions, and events. We'll keep you in the loop and make sure you never miss out on a great deal.

Thank you for choosing us, Ahmed. We look forward to serving you and helping you find the perfect products for your needs.

Best regards,
[Your Name]

In [ ]:
# Scoring Agent
